In [1]:
# Cell 1 — Run Step 2 + Step 3 on a single query and print the full
# JSON results from both stages. Cell 2 reads the globals stamped at
# the bottom of this cell to drive a single category-handler call.
#
# Setup (imports, .env, DB pools) is folded in here so the notebook
# is exactly two cells per the testing contract.

import sys, json, asyncio, time
from pathlib import Path

project_root = str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from dotenv import load_dotenv
load_dotenv(Path(project_root) / ".env")

# Pipeline runners.
from search_v2.step_2 import run_step_2
from search_v2.step_3 import run_step_3

# Database connections — mirrors the FastAPI lifespan in api/main.py
# so the handler / executors in Cell 2 have working pool clients.
from db.postgres import pool as postgres_pool
from db.redis import init_redis, check_redis
from db.qdrant import check_qdrant, qdrant_client
import db.redis as _redis_module

# Idempotent: rerunning this cell must not double-open the Postgres
# pool or leak a prior Redis client.
if postgres_pool._closed:
    await postgres_pool.open()
    print("Postgres: pool opened")
else:
    print("Postgres: pool already open")

if _redis_module._redis_client is None:
    await init_redis()
    print(f"Redis:    initialized ({await check_redis()})")
else:
    print(f"Redis:    already initialized ({await check_redis()})")

print(f"Qdrant:   {await check_qdrant()}")
print()

/Users/michaelkeohane/Documents/movie-finder-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Postgres: pool opened
Redis:    initialized (ok)
Qdrant:   ok



In [2]:
# ----------------------------------------------------------------
# QUERY — edit this to test a different input.
# ----------------------------------------------------------------
QUERY = (
    "good for my dog"
)
print(f"[query] {QUERY}\n")

# ----------------------------------------------------------------
# Step 2 — atoms + traits.
# ----------------------------------------------------------------
step_2_start = time.perf_counter()
STEP_2_ANALYSIS, step_2_in, step_2_out, step_2_elapsed = await run_step_2(QUERY)
print("=" * 70)
print("STEP 2 — QueryAnalysis")
print("=" * 70)
print(json.dumps(STEP_2_ANALYSIS.model_dump(), indent=2, ensure_ascii=False, default=str))
print(
    f"\n[step 2 stats] elapsed={step_2_elapsed:.2f}s "
    f"input_tokens={step_2_in} output_tokens={step_2_out}"
)

# ----------------------------------------------------------------
# Step 3 — per-trait decomposition, fanned out in parallel.
# STEP_3_RESULTS is a list aligned with STEP_2_ANALYSIS.traits:
# each entry is the TraitDecomposition for traits[i]. Cell 2 reads
# this list to pick a CategoryCall by (trait_index, call_index).
# ----------------------------------------------------------------
print("\n" + "=" * 70)
print("STEP 3 — TraitDecomposition per trait")
print("=" * 70)

if not STEP_2_ANALYSIS.traits:
    STEP_3_RESULTS = []
    print("(no traits emitted by Step 2 — nothing to decompose)")
else:
    # Each Step 3 call receives its sibling traits' structural fields
    # so positioning references can drop axes that siblings replace.
    raw = await asyncio.gather(
        *(
            run_step_3(
                trait,
                [s for s in STEP_2_ANALYSIS.traits if s is not trait],
            )
            for trait in STEP_2_ANALYSIS.traits
        )
    )
    # Drop token / timing tuples — keep just the decompositions, aligned
    # positionally with STEP_2_ANALYSIS.traits.
    STEP_3_RESULTS = [decomp for decomp, _in, _out, _t in raw]

    step_3_in_total = sum(in_tok for _d, in_tok, _o, _t in raw)
    step_3_out_total = sum(out_tok for _d, _i, out_tok, _t in raw)
    step_3_wallclock = max((t for _d, _i, _o, t in raw), default=0.0)

    for idx, (trait, decomp) in enumerate(zip(STEP_2_ANALYSIS.traits, STEP_3_RESULTS)):
        print(f'\n--- trait[{idx}]: "{trait.surface_text}" ---')
        print(json.dumps(decomp.model_dump(), indent=2, ensure_ascii=False, default=str))

    print(
        f"\n[step 3 stats] wallclock={step_3_wallclock:.2f}s "
        f"(slowest of {len(raw)} parallel calls) "
        f"input_tokens={step_3_in_total} output_tokens={step_3_out_total}"
    )

# ----------------------------------------------------------------
# Index summary — quick reference for Cell 2's (trait_idx, call_idx).
# ----------------------------------------------------------------
print("\n" + "=" * 70)
print("CATEGORY-CALL INDEX (use these in Cell 2)")
print("=" * 70)
for t_idx, (trait, decomp) in enumerate(zip(STEP_2_ANALYSIS.traits, STEP_3_RESULTS)):
    print(f'  trait[{t_idx}] "{trait.surface_text}"  '
          f'(polarity={trait.polarity.value}, role={trait.relationship_role.value}, '
          f'combine_mode={decomp.combine_mode.value})')
    for c_idx, call in enumerate(decomp.category_calls):
        print(f'    call[{c_idx}] category={call.category.name}  expressions={call.expressions}')


[query] good for my dog

[Gemini] cached_content_token_count=0 (prompt=11716)
STEP 2 — QueryAnalysis
{
  "intent_exploration": "The query 'good for my dog' is most likely an experiential request for movies that are safe or enjoyable to watch with a pet present (e.g., no high-pitched distress frequencies, no aggressive barking, or visually engaging for canines). A less likely but plausible read is a request for movies ABOUT dogs, though the phrasing 'good for' typically implies a viewing occasion or audience fit rather than a subject matter. The most likely intent is a viewing occasion where the movie's content is compatible with a dog's presence in the room.",
  "atoms": [
    {
      "surface_text": "good",
      "modifying_signals": [
        {
          "surface_phrase": "for my dog",
          "effect": "Qualifies the viewing occasion and target audience, shaping the evaluation toward pet-friendly content."
        }
      ],
      "evaluative_intent": "Movies that are suitable or 

In [12]:
# Cell 2 — Run the V3 category handler on ONE CategoryCall and print
# the top 25 scored candidates per fired endpoint spec.
#
# Two ways to pick the call:
#  (a) by index into Cell 1's results — set TRAIT_INDEX / CALL_INDEX.
#  (b) manually — set USE_MANUAL = True and edit the manual block.
# (b) is useful when iterating on the handler without re-running Step 2/3.

import json, time
from datetime import datetime, timezone

from db.postgres import fetch_movie_cards
from db.qdrant import qdrant_client
from schemas.enums import Polarity, TraitRelationshipRole
from schemas.step_2 import Trait
from schemas.step_3 import CategoryCall
from schemas.trait_category import CategoryName
from search_v2.endpoint_fetching.category_handlers.handler import (
    run_query_generation,
)
from search_v2.endpoint_fetching.endpoint_executors import (
    build_endpoint_coroutine,
)

TOP_N = 25

# ----------------------------------------------------------------
# Selection mode
# ----------------------------------------------------------------
USE_MANUAL = True  # flip to True to use the manual block below

# Index-mode picker — refers to the CATEGORY-CALL INDEX printed at
# the bottom of Cell 1.
TRAIT_INDEX = 0
CALL_INDEX = 0

if not USE_MANUAL:
    if "STEP_2_ANALYSIS" not in globals() or "STEP_3_RESULTS" not in globals():
        raise RuntimeError(
            "Run Cell 1 first so STEP_2_ANALYSIS and STEP_3_RESULTS exist, "
            "or set USE_MANUAL = True and fill in the manual block below."
        )
    trait = STEP_2_ANALYSIS.traits[TRAIT_INDEX]
    decomp = STEP_3_RESULTS[TRAIT_INDEX]
    category_call = decomp.category_calls[CALL_INDEX]
    sibling_calls = [c for i, c in enumerate(decomp.category_calls) if i != CALL_INDEX]
    combine_mode = decomp.combine_mode
    selection_label = (
        f"trait[{TRAIT_INDEX}]={trait.surface_text!r}  "
        f"call[{CALL_INDEX}]={category_call.category.name}"
    )
else:
    # ------------------------------------------------------------
    # Manual block — construct a Trait + CategoryCall directly.
    # Adjust the surface_text / category / expressions / role fields
    # to exercise the handler without going through Step 2/3.
    # ------------------------------------------------------------
    trait = Trait(
        surface_text="'The rock' movies",
        evaluative_intent=(
            "The user wants movies where 'the rock' is a lead actor."
        ),
        qualifier_relation="n/a",
        relationship_role=TraitRelationshipRole.INDEPENDENT,
        replaces_axis=None,
        axes_replaced_by_siblings=[],
        anchor_reference="n/a",
        polarity=Polarity.POSITIVE,
        commitment_evidence="Manually constructed for this notebook probe.",
        commitment="required",
        contextualized_phrase="animated movies for kids",
    )
    category_call = CategoryCall(
        category=CategoryName.SEASONAL_HOLIDAY,
        expressions=["christmas"],
        retrieval_intent=(
            "Find Christmas movies."
        ),
    )
    sibling_calls = []
    combine_mode = None  # let the handler default to single-call mode
    selection_label = f"manual / {category_call.category.name}"

print(f"[selection] {selection_label}")
print(f"[category_call] {category_call.model_dump_json(indent=2)}")
print()

# ----------------------------------------------------------------
# 1) Query generation — the handler LLM emits 0+ endpoint specs.
# ----------------------------------------------------------------
gen_start = time.perf_counter()
specs = await run_query_generation(
    category_call=category_call,
    trait=trait,
    sibling_calls=sibling_calls,
    combine_mode=combine_mode,
)
gen_elapsed = time.perf_counter() - gen_start
print(f"Generated {len(specs)} endpoint spec(s) in {gen_elapsed:.2f}s\n")

# GeneratedEndpointSpec is a dataclass with a pydantic-model `params`
# field, so we build a JSON-friendly dict by hand: dataclass fields
# get their enum .value, and `params` is dumped via model_dump() so
# every nested reasoning/exploration field is included.
for i, spec in enumerate(specs):
    print("=" * 60)
    print(f"SPEC #{i + 1}")
    print("=" * 60)
    spec_dict = {
        "route": spec.route.value,
        "operation_type": spec.operation_type.value,
        "was_promoted": spec.was_promoted,
        "params": spec.params.model_dump() if spec.params is not None else None,
    }
    print(json.dumps(spec_dict, indent=2, ensure_ascii=False, default=str))
    print()

# ----------------------------------------------------------------
# 2) Execution — fire each spec against the full corpus and show the
# top 25 scored candidates (resolved to title + year via a single
# bulk movie_card fetch).
# ----------------------------------------------------------------
for i, spec in enumerate(specs):
    print("=" * 60)
    print(f"EXECUTION #{i + 1} — {spec.route.value}")
    print("=" * 60)
    exec_start = time.perf_counter()
    result = await build_endpoint_coroutine(
        spec,
        qdrant_client=qdrant_client,
        restrict_to_movie_ids=None,
    )
    exec_elapsed = time.perf_counter() - exec_start
    print(f"executed in {exec_elapsed:.2f}s — {len(result.scores)} scored candidates")

    top = sorted(result.scores, key=lambda c: c.score, reverse=True)[:TOP_N]
    if not top:
        print("  (no scored candidates)\n")
        continue

    # Single bulk Postgres fetch — never query per-candidate.
    cards = await fetch_movie_cards([c.movie_id for c in top])
    card_by_id = {c["movie_id"]: c for c in cards}
    for c in top:
        card = card_by_id.get(c.movie_id)
        if card is None:
            title, year = "<missing card>", "?"
        else:
            title = card["title"] or "<untitled>"
            ts = card["release_ts"]
            year = (
                datetime.fromtimestamp(ts, tz=timezone.utc).year
                if ts is not None else "?"
            )
        print(f"  {c.score:.3f}  {title} ({year})  [movie_id={c.movie_id}]")
    print()


[selection] manual / SEASONAL_HOLIDAY
[category_call] {
  "category": "Seasonal / holiday",
  "expressions": [
    "christmas"
  ],
  "retrieval_intent": "Find Christmas movies."
}

Generated 1 endpoint spec(s) in 3.66s

SPEC #1
{
  "route": "semantic",
  "operation_type": "pool_reranker",
  "was_promoted": false,
  "params": {
    "parameters": {
      "role_exploration": "This is a population-naming seasonal query: movies that match Christmas framing directly, not movies compared against a reference.",
      "role": "carver",
      "aspects": [
        "Christmas viewing occasion",
        "Christmas-set stories"
      ],
      "space_candidates": [
        {
          "space": "watch_context",
          "strengths": "Christmas viewing occasion; holiday movie night; family holiday watching; seasonal watch scenarios",
          "weaknesses": "under-coverage: Christmas-set story content is not here; over-coverage: can pull general holiday occasions beyond Christmas"
        },
        